# Diet Adherence Model Training

This notebook trains a Support Vector Regression (SVR) model to predict a diet adherence score based on lifestyle factors.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Dataset

In [ ]:
# Load the dataset from the current folder
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
df.head()

## 2. Select Features & Create Target Variable

We use `Sleep Duration`, `Physical Activity Level`, `Stress Level`, and `Daily Steps`.
Since we don't have a direct adherence score, we will synthesize one from these variables.

In [ ]:
features = ['Sleep Duration', 'Physical Activity Level', 'Stress Level', 'Daily Steps']
X_raw = df[features].copy()

# Normalize features to 0-1 scale to construct the synthetic target
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=features)

# Adherence increases with Sleep, Physical Activity, Daily Steps, and decreases with Stress
df['adherence_score'] = (
    X_scaled['Sleep Duration'] + 
    X_scaled['Physical Activity Level'] + 
    X_scaled['Daily Steps'] + 
    (1 - X_scaled['Stress Level'])
) / 4

print("Adherence score summary:")
print(df['adherence_score'].describe())

## 3. Train-Test Split

In [ ]:
X = df[features]
y = df['adherence_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

## 4. Train the Support Vector Regression (SVR) Model
Note: We are using a `StandardScaler` to scale the inputs before sending them to the SVR model. SVR performs poorly if inputs aren't scaled properly.

In [ ]:
model = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.01))
model.fit(X_train, y_train)

## 5. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R\u00b2 Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")

## 6. Export the Model

In [ ]:
with open('diet_adherence_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model successfully saved to 'diet_adherence_model.pkl'")

## 7. Example Prediction

In [ ]:
example_data = {
    'Sleep Duration': [6.0], 
    'Physical Activity Level': [30], 
    'Stress Level': [7], 
    'Daily Steps': [4000]
}
example_df = pd.DataFrame(example_data)

prediction = model.predict(example_df)[0]
print(f"Predicted Adherence Score for average user: {prediction:.4f}")

example_data_good = {
    'Sleep Duration': [8.0], 
    'Physical Activity Level': [80], 
    'Stress Level': [3], 
    'Daily Steps': [10000]
}
example_df_good = pd.DataFrame(example_data_good)

prediction_good = model.predict(example_df_good)[0]
print(f"Predicted Adherence Score for highly active user: {prediction_good:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='green')
min_val = min(np.min(y_test), np.min(y_pred))
max_val = max(np.max(y_test), np.max(y_pred))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
plt.title('Actual vs Predicted - Diet Adherence Score')
plt.xlabel('Actual Score')
plt.ylabel('Predicted Score')
plt.grid(True)
plt.tight_layout()
plt.savefig('diet_adherence_predictions.png', dpi=300)
plt.show()